In [5]:
import pandas as pd
import numpy as np
import json, warnings
warnings.filterwarnings("ignore")

In [6]:
df = pd.read_csv(r"C:\Users\Kalyan\OneDrive\Desktop\ecom_sales_analysis\ecommerce_sales.csv", parse_dates=["Order_Date", "Ship_Date"])

In [7]:
df.shape
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Order_ID          20000 non-null  object        
 1   Order_Date        20000 non-null  datetime64[ns]
 2   Ship_Date         20000 non-null  datetime64[ns]
 3   Ship_Mode         20000 non-null  object        
 4   Customer_ID       20000 non-null  object        
 5   Customer_Segment  20000 non-null  object        
 6   Region            20000 non-null  object        
 7   Category          20000 non-null  object        
 8   Unit_Price        20000 non-null  float64       
 9   Quantity          20000 non-null  int64         
 10  Discount          20000 non-null  float64       
 11  Sales             20000 non-null  float64       
 12  Profit            20000 non-null  float64       
 13  Profit_Margin     20000 non-null  float64       
 14  Payment_Method    2000

In [8]:
# --Data cleaning--
df.isnull().sum()
df.drop_duplicates(inplace=True)
df.dtypes


Order_ID                    object
Order_Date          datetime64[ns]
Ship_Date           datetime64[ns]
Ship_Mode                   object
Customer_ID                 object
Customer_Segment            object
Region                      object
Category                    object
Unit_Price                 float64
Quantity                     int64
Discount                   float64
Sales                      float64
Profit                     float64
Profit_Margin              float64
Payment_Method              object
Year                         int64
Month                        int64
Quarter                     object
Discount_Bucket             object
dtype: object

In [6]:
df.head(10)

,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Segment,Region,Category,Unit_Price,Quantity,Discount,Sales,Profit,Profit_Margin,Payment_Method,Year,Month,Quarter,Discount_Bucket
0,ORD-100001,2022-08-17,2022-08-21,Express,CUST-12848,Consumer,North,Electronics,774.16,2,0.04,1486.39,244.32,0.1644,UPI,2022,8,Q3,Low (1-10%)
1,ORD-100002,2024-05-14,2024-05-18,Express,CUST-31295,Consumer,North,Electronics,1151.09,7,0.35,5237.46,52.37,0.0100,UPI,2024,5,Q2,Very High (31%+)
2,ORD-100003,2024-05-09,2024-05-16,Standard,CUST-23848,Home Office,East,Clothing,67.32,1,0.35,43.76,8.27,0.1889,Net Banking,2024,5,Q2,Very High (31%+)
3,ORD-100004,2023-11-28,2023-12-04,Standard,CUST-37770,Corporate,North,Clothing,151.43,2,0.08,278.63,99.34,0.3565,Debit Card,2023,11,Q4,Low (1-10%)
4,ORD-100005,2023-12-06,2023-12-09,Express,CUST-12582,Corporate,West,Beauty & Health,17.89,10,0.18,146.70,60.35,0.4114,COD,2023,12,Q4,Medium (11-20%)
5,ORD-100006,2023-08-24,2023-08-26,Express,CUST-31668,Home Office,East,Books & Media,31.66,5,0.23,121.89,33.49,0.2748,UPI,2023,8,Q3,High (21-30%)
6,ORD-100007,2023-04-12,2023-04-13,Same-Day,CUST-30830,Home Office,West,Sports & Outdoor,279.15,7,0.17,1621.86,353.12,0.2177,Debit Card,2023,4,Q2,Medium (11-20%)
7,ORD-100008,2024-01-18,2024-01-20,Express,CUST-29960,Corporate,South,Electronics,535.70,1,0.03,519.63,86.28,0.1660,UPI,2024,1,Q1,Low (1-10%)
8,ORD-100009,2022-12-17,2022-12-24,Standard,CUST-28250,Consumer,North,Home & Furniture,127.27,4,0.23,391.99,64.99,0.1658,Wallet,2022,12,Q4,High (21-30%)
9,ORD-100010,2023-03-26,2023-04-01,Standard,CUST-23145,Corporate,West,Beauty & Health,24.35,1,0.08,22.40,10.97,0.4897,Credit Card,2023,3,Q1,Low (1-10%)


In [9]:
# ── 1. Basic Cleaning  ────────────────────────────────
df["Ship_Days"] = (df["Ship_Date"] - df["Order_Date"]).dt.days
df["Revenue_Per_Unit"] = (df["Sales"] / df["Quantity"]).round(2)
df["Is_Profitable"] = df["Profit"] > 0
df["YearMonth"] = df["Order_Date"].dt.to_period("M").astype(str)

In [8]:
df.head(5)

,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Segment,Region,Category,Unit_Price,Quantity,...,Profit_Margin,Payment_Method,Year,Month,Quarter,Discount_Bucket,Ship_Days,Revenue_Per_Unit,Is_Profitable,YearMonth
0,ORD-100001,2022-08-17,2022-08-21,Express,CUST-12848,Consumer,North,Electronics,774.16,2,...,0.1644,UPI,2022,8,Q3,Low (1-10%),4,743.20,True,2022-08
1,ORD-100002,2024-05-14,2024-05-18,Express,CUST-31295,Consumer,North,Electronics,1151.09,7,...,0.0100,UPI,2024,5,Q2,Very High (31%+),4,748.21,True,2024-05
2,ORD-100003,2024-05-09,2024-05-16,Standard,CUST-23848,Home Office,East,Clothing,67.32,1,...,0.1889,Net Banking,2024,5,Q2,Very High (31%+),7,43.76,True,2024-05
3,ORD-100004,2023-11-28,2023-12-04,Standard,CUST-37770,Corporate,North,Clothing,151.43,2,...,0.3565,Debit Card,2023,11,Q4,Low (1-10%),6,139.32,True,2023-11
4,ORD-100005,2023-12-06,2023-12-09,Express,CUST-12582,Corporate,West,Beauty & Health,17.89,10,...,0.4114,COD,2023,12,Q4,Medium (11-20%),3,14.67,True,2023-12


In [10]:
# — Exploratory Data Analysis (EDA)--
df["Sales"].sum()

np.float64(12209063.57)

In [11]:
df["Profit"].sum()

np.float64(1882030.23)

In [12]:
# -Sales by category
df.groupby("Category")["Sales"].sum().sort_values(ascending=False)

Category
Electronics         7194776.31
Home & Furniture    2565594.17
Sports & Outdoor    1030810.76
Clothing             707247.93
Beauty & Health      319474.49
Toys & Games         238401.04
Books & Media         94787.55
Groceries             57971.32
Name: Sales, dtype: float64

In [13]:
#--Profit by region
df.groupby("Region")["Profit"].sum()

Region
East     463727.37
North    542018.88
South    409590.34
West     466693.64
Name: Profit, dtype: float64

In [14]:
#--Top customers
df.groupby("Customer_ID")["Sales"].sum().sort_values(ascending=False).head(10)

Customer_ID
CUST-12816    21579.67
CUST-22216    21427.11
CUST-13725    16500.38
CUST-32993    16041.46
CUST-10223    15550.08
CUST-36538    15129.84
CUST-36878    14646.60
CUST-29299    14639.51
CUST-37609    14633.06
CUST-13090    14601.19
Name: Sales, dtype: float64

In [24]:
#-Monthly sales
df.groupby("YearMonth")["Sales"].sum()

YearMonth
2022-01    336646.63
2022-02    217291.86
2022-03    346363.70
2022-04    358094.41
2022-05    287628.15
2022-06    311852.23
2022-07    295977.23
2022-08    332783.74
2022-09    300925.38
2022-10    387066.96
2022-11    370563.78
2022-12    474884.82
2023-01    394678.57
2023-02    261061.81
2023-03    284086.15
2023-04    346847.45
2023-05    381658.99
2023-06    271686.62
2023-07    291586.60
2023-08    363388.26
2023-09    353603.22
2023-10    327872.53
2023-11    377666.37
2023-12    389218.53
2024-01    362601.11
2024-02    248938.53
2024-03    348074.71
2024-04    336187.50
2024-05    351172.65
2024-06    294400.57
2024-07    333201.32
2024-08    310775.52
2024-09    394794.23
2024-10    336912.41
2024-11    369830.82
2024-12    458740.21
Name: Sales, dtype: float64

In [2]:
top_customers = df.groupby("Customer_ID")["Sales"].sum().sort_values(ascending=False).head(10)



NameError: name 'df' is not defined

In [15]:
#--Exporting cleaned data
df.to_csv("data_jpyn/cleaned_ecommerce_sales.csv", index=False)

In [21]:
df.to_csv("C:/Users/Kalyan/Desktop/cleaned_ecommerce_sales.csv", index=False)

In [19]:
import os

os.makedirs("C:/Users/Kalyan/Desktop/ecom_sales_analysis", exist_ok=True)

df.to_csv("C:/Users/Kalyan/Desktop/ecom_sales_analysis/cleaned_ecommerce_sales.csv", index=False)